# Vibrato: real annotations and controlled injection

Two complementary comparisons for choosing Attune's vibrato estimator. Yang tests agreement with annotated real performances; injected CocoChorales tests recovery of known time-varying modulation through rendered audio and the pitch frontend. Synthetic ground truth is informative about curve estimation, while conclusions remain scoped to the renderer, profiles, and sampled instruments.

Each table contains **one Attune row**: current production note detection → score alignment/timing refit → repeat recovery, followed by the production vibrato estimator and a benchmark-only frame gate copied from Yang's decision-tree thresholds: **4–9 Hz, inclusive, and ≥0.10 semitones one-sided extent (20 cents peak-to-peak)**. The gate intersects existing detections; it does not alter fitted curves. No extra quality or minimum-interval gate is applied. This does not reproduce Yang's complete decision cleanup or change app defaults.

Run with the repository `at-venv` kernel. Each run cell is self-contained, reuses matching checkpoints, and writes source-keyed `paired_*` reports. The Yang cell requires the existing prepared candidate score assets in `benchmarks/results/yang_score_boundary_v1_inputs`.


## Yang vibrato dataset

The [released corpus](https://github.com/skx300/vibrato_dataset) contains vibrato-region annotations and, for a small erhu/violin subset, manually marked half cycles. This run includes **all six local parameter-annotated audio files**, rather than selecting two. These are excerpt files, not six independent compositions or performers. The [paper](https://luweiyangqm.files.wordpress.com/2017/05/jmm2017_luwei_yang.pdf), §§4.1–4.3, describes the parameter corpus around *The Moon Reflected on the Second Spring*: musical diversity is narrow. The local candidate-score mapping is unverified and does not independently establish the piece identity of every excerpt.

Interpretation limits:

- **Event-averaged parameter targets.** Following Yang’s original parameter benchmark, rate and extent are the arithmetic means of the annotated half-cycle estimates within each vibrato event. These means are used as constant targets across the event, even when the performance varies. This evaluates event-level parameter estimates rather than recovery of within-event curves. CocoChorales retains its time-varying injected targets.
- **Vibrato regions are not musical-note boundaries.** Vibrato may start or stop inside a note. Attune receives production score-aware note boundaries, never the annotated vibrato spans as analysis cuts. Prepared symbolic score excerpts were selected using pitch-based matching; the final boundaries are recomputed by `Recording.analyze_notes()`, not taken from DTW. Candidate arrangements/matches remain an additional source of uncertainty.
- **Non-vibrato does not mean constant pitch.** Pitch bends/portamento can occur outside annotated vibrato regions. These remain negative examples, including the bends observed during inspection of this corpus; their prevalence has not been separately quantified. This tests rejection of non-oscillatory pitch movement as well as vibrato detection.
- **Limited independent parameter truth.** Only six local files have half-cycle annotations. In this adapter, rate uses manual extrema timing, but extent uses the common estimated pitch at those markers, so extent truth is not independent of the pitch frontend.
- **Training provenance.** The released Yang Bayes model has provenance in this corpus; its performance here is not evidence of transfer to unseen performers.

All methods receive the same complete pitch contour and are scored over complete recordings, including negative regions. Frame F1 measures presence; rate/extent F1 require detection and estimates within 0.5 Hz / 0.05 semitones one-sided extent. The soft parameter score gives partial credit. Yang event F1 evaluates annotated vibrato intervals, not musical-note segmentation. A lower event score alone does not establish a worse per-note parameter curve.


In [ ]:
from pathlib import Path
import sys

ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
            if (p / 'app.py').is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from benchmarks.modules.vibrato.VibratoNotebook import VibratoNotebook

yang_results, yang_output = VibratoNotebook.run_suite('yang', root=ROOT, workers=2)
VibratoNotebook.show_results(yang_results)
print(f'Reports: {yang_output}')


### Paired significance: Yang

The requested six recording files are paired units (64 exact label swaps). They share performers and repertoire, so this is exploratory excerpt-level inference, not six independent performances. A separate two-performer sensitivity table is saved; neither analysis establishes generalization to new compositions.

As in the pitch/note benchmarks, scores are recomputed from pooled counts in each resample. Two-sided paired method-label swaps test the difference; 9,999 cluster bootstrap draws give 95% percentile confidence intervals. All seven metrics × all requested competitors form one Holm family per dataset (14 Yang tests; 28 Coco tests). Positive differences favor Attune. Confidence intervals are unadjusted and need not agree with the corrected significance flag. These are exploratory tests selected after inspecting the results. They reuse saved checkpoints; no audio inference is rerun.

With six paired units, the smallest two-sided exact p-value is 0.03125: no Yang test can pass this 14-test Holm family at 0.05. More independent data—not more bootstrap draws—would improve resolution.

In [ ]:
from pathlib import Path
import sys

ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
            if (p / 'app.py').is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from benchmarks.modules.vibrato.VibratoNotebook import VibratoNotebook

# Use the paired checkpoint reports from the suite above.
source_run = yang_output
yang_tests, yang_tests_output = VibratoNotebook.run_significance(source_run, 'yang')
VibratoNotebook.show_significance(yang_tests)
print(f'Reports: {yang_tests_output}')
import pandas as pd
print('Performer-grouped sensitivity (two groups; limited inferential resolution):')
VibratoNotebook.show_significance(pd.read_csv(yang_tests_output / 'performer_sensitivity.csv'))


## CocoChorales vibrato injection

The preliminary run selects **20 test-split stems**, balanced across ensemble/instrument groups with seed 0. It injects up to six eligible notes per stem (minimum duration 0.75 s), assigning the profiles below reproducibly. Stems with fewer eligible notes contribute fewer profiles; saved case reports show actual coverage.

| Profile | Modulation |
|---|---|
| Control (`none`) | Straight tone; no injected vibrato |
| Constant | Fixed rate and extent |
| Accelerating | Increasing rate |
| Decelerating | Decreasing rate |
| Widening | Increasing extent |
| Narrowing | Decreasing extent |

Rates span **3–10 Hz** and one-sided extents **0.15–1.00 semitones**. Changing rates span 1–3 Hz; changing extents span 0.15–0.50 semitones, within those bounds. Each rendered stem is evaluated clean and with **20 dB / 10 dB** noise. The benchmark-specific sfizz straight-tone policy suppresses native patch vibrato. The shared Attune pitch frontend processes rendered audio; each method receives the same continuous contour, including note transitions. Production score-aware segmentation is performed independently for each audio condition. Injection timestamps are evaluation truth, not supplied analysis boundaries.

The commanded rate and extent are known at each time, allowing direct evaluation of constant and changing curves plus false positives on controls. MIDI bend sampling/quantization, synthesis, and pitch tracking still affect the observed signal; commanded modulation is not a separately measured acoustic F0 reference. Repeated noise conditions and injected notes are not independent performances.

**The same 4–9 Hz gate is used here:** true injected vibrato outside that range remains in the benchmark and can count as missed. This deliberately preserves the original 3–10 Hz task; the run is not restricted to Yang's preferred range. Interpret any losses outside the gate accordingly.

The table uses the same detection and parameter metrics as above, now against the time-varying injected curves. These results directly inform estimator choice for the specified modulation profiles, complementing the real-data comparison rather than replacing it.


In [ ]:
from pathlib import Path
import sys

ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
            if (p / 'app.py').is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from benchmarks.modules.vibrato.VibratoNotebook import VibratoNotebook

coco_results, coco_output = VibratoNotebook.run_suite('coco', root=ROOT, workers=2)
VibratoNotebook.show_results(coco_results)
print(f'Reports: {coco_output}')


### Paired significance: CocoChorales

The 20 source tracks are paired clusters. All injected notes and clean/20 dB/10 dB versions of a source stay together. Bootstrap sampling is stratified by ensemble category. The 318 cases are not 318 independent observations.

Scores are recomputed from pooled counts. Two-sided paired method-label swaps use 9,999 draws, with the +1 Monte Carlo correction; 9,999 ensemble-stratified cluster bootstrap draws give 95% percentile intervals. All seven metrics × four competitors form one separate 28-test Holm family. Positive differences favor Attune. Intervals are unadjusted; significance uses corrected p-values. This exploratory analysis reuses checkpoints without rerunning estimators.

In [ ]:
from pathlib import Path
import sys

ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
            if (p / 'app.py').is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from benchmarks.modules.vibrato.VibratoNotebook import VibratoNotebook

# Use the paired checkpoint reports from the suite above.
source_run = coco_output
coco_tests, coco_tests_output = VibratoNotebook.run_significance(source_run, 'coco')
VibratoNotebook.show_significance(coco_tests)
print(f'Reports: {coco_tests_output}')
